In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('../data/adn_boca_real_features.csv')

print(f"Dataset: {df.shape}")
print(df.isnull().sum())

In [ ]:
le = LabelEncoder()
df['posicion_encoded'] = le.fit_transform(df['posicion'])

# rating NO se usa porque la etiqueta se definió en base a rating (data leakage)
features = ['posicion_encoded', 'edad', 'partidos', 
            'goles', 'asistencias', 'pases_precisos',
            'participacion_gol', 'goles_por_partido']

X = df[features]
y = df['etiqueta']

print("Distribución de etiquetas:")
print(y.value_counts())
print(f"\nFeatures: {features}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
import joblib

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

modelo = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=5,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42
)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
print("📊 Resultados del modelo (con validación cruzada):")
print(classification_report(y_test, y_pred, 
      target_names=['No encaja', 'Perfil Boca']))

cv_scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
print(f"CV Accuracy: {cv_scores.mean():.2f} (+/- {cv_scores.std() * 2:.2f})")

joblib.dump(modelo, '../models/modelo_adn_boca.pkl')
joblib.dump(le, '../models/label_encoder.pkl')
print("\n✅ Modelo guardado en /models/")

In [ ]:
import matplotlib.pyplot as plt

importancias = pd.Series(
    modelo.feature_importances_, 
    index=features
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importancias.plot(kind='barh', ax=ax, color='#003087')  # Azul Boca
ax.set_title('¿Qué define el ADN Boca?', fontsize=14, fontweight='bold')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.savefig('../outputs/importancia_features.png', dpi=150)
plt.show()
print("✅ Gráfico guardado")

In [ ]:

df_plantilla = pd.read_csv('../data/adn_boca_real_features.csv')

df_plantilla['posicion_encoded'] = df_plantilla['posicion'].apply(
    lambda x: le.transform([x])[0] if x in le.classes_ else 0
)

X_plantilla = df_plantilla[features]

# Predicción
df_plantilla['encaja_boca'] = modelo.predict(X_plantilla)
df_plantilla['probabilidad'] = modelo.predict_proba(X_plantilla)[:, 1]

resultado = df_plantilla[['nombre', 'posicion', 'partidos', 'goles', 
                           'probabilidad', 'encaja_boca']]
resultado = resultado.sort_values('probabilidad', ascending=False)

print("🏆 Top 10 jugadores con perfil Boca:")
print(resultado.head(10).to_string(index=False))

resultado.to_csv('../data/scouting_resultado.csv', index=False)
print("\n✅ Resultados guardados en data/scouting_resultado.csv")